# Naive Bayes: Predicting survival from titanic crash

In [21]:
import pandas as pd

In [86]:
df = pd.read_csv("data_titanic.csv")
df.head()

,PassengerId,Name,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,1,"Braund, Mr. Owen Harris",3,male,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,"Heikkinen, Miss. Laina",3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,female,35.0,1,0,113803,53.1000,C123,S,1
4,5,"Allen, Mr. William Henry",3,male,35.0,0,0,373450,8.0500,NaN,S,0


In [87]:
df.shape

(891, 12)

In [88]:
# Lets drop columns that we do not need
df.drop(['PassengerId', # Just an index, no predictive value.
         'Name', # Not predictive except titles, but titles can be extracted separately (optional).
         'Ticket', # Not helpful for prediction.
         'SibSp','Parch', 'Embarked', # drop these too to keep this example lite
        ],
        axis='columns',inplace=True)
df.head()

,Pclass,Sex,Age,Fare,Cabin,Survived
0,3,male,22.0,7.2500,NaN,0
1,1,female,38.0,71.2833,C85,1
2,3,female,26.0,7.9250,NaN,1
3,1,female,35.0,53.1000,C123,1
4,3,male,35.0,8.0500,NaN,0


In [89]:
# missing values
df.isnull().sum()

Pclass        0
Sex           0
Age         177
Fare          0
Cabin       687
Survived      0
dtype: int64

In [90]:
# Percent of missing values
missing = (df.isnull().sum() / len(df)) * 100
missing = missing.to_frame(name='percent_missing')
print(missing)

          percent_missing
Pclass           0.000000
Sex              0.000000
Age             19.865320
Fare             0.000000
Cabin           77.104377
Survived         0.000000


In [91]:
# Cabin is missing 77% of data. Drop the column.
df.drop(['Cabin'], axis='columns',inplace=True)
df.head()

,Pclass,Sex,Age,Fare,Survived
0,3,male,22.0,7.2500,0
1,1,female,38.0,71.2833,1
2,3,female,26.0,7.9250,1
3,1,female,35.0,53.1000,1
4,3,male,35.0,8.0500,0


In [97]:
# seperate features and target
X = df.drop('Survived',axis='columns') # features
y = df['Survived'] # targets

In [98]:
# map male -> 0 and female -> 1
X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})

In [84]:
# Find features that have null values
X.columns[X.isna().any()]

Index(['Age'], dtype='object')

In [66]:
# Give me the count 
X.isnull().sum()

Pclass      0
Sex         0
Age       177
Fare        0
dtype: int64

In [59]:
# Lets replace that with mean
X['Age'] = X['Age'].fillna(X['Age'].mean())
X.head()

,Pclass,Sex,Age,Fare
0,3,0,22.0,7.2500
1,1,1,38.0,71.2833
2,3,1,26.0,7.9250
3,1,1,35.0,53.1000
4,3,0,35.0,8.0500


In [49]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [99]:
print(X_train.shape)
print(X_test.shape)

(623, 4)
(268, 4)


In [50]:
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()

In [51]:
model.fit(X_train, y_train)

GaussianNB()

In [53]:
model.score(X_test, y_test)

0.7835820895522388

In [54]:
X_test[0:10]

,Pclass,Sex,Age,Fare
64,1,0,29.699118,27.7208
832,3,0,29.699118,7.2292
838,3,0,32.000000,56.4958
719,3,0,33.000000,7.7750
560,3,0,29.699118,7.7500
203,3,0,45.500000,7.2250
367,3,1,29.699118,7.2292
884,3,0,25.000000,7.0500
754,2,1,48.000000,65.0000
303,2,1,29.699118,12.3500


In [55]:
y_test[0:10]

64     0
832    0
838    1
719    0
560    0
203    0
367    1
884    0
754    1
303    1
Name: Survived, dtype: int64

In [56]:
model.predict(X_test[0:10])

array([0, 0, 0, 0, 0, 0, 1, 0, 1, 1], dtype=int64)

In [57]:
model.predict_proba(X_test[:10])

array([[0.7855139 , 0.2144861 ],
       [0.96683984, 0.03316016],
       [0.9237868 , 0.0762132 ],
       [0.96821285, 0.03178715],
       [0.96699329, 0.03300671],
       [0.96884274, 0.03115726],
       [0.42756018, 0.57243982],
       [0.96418444, 0.03581556],
       [0.0914496 , 0.9085504 ],
       [0.27525075, 0.72474925]])

**Calculate the score using cross validation**

In [58]:
from sklearn.model_selection import cross_val_score
cross_val_score(GaussianNB(), X_train, y_train, cv=5)

array([0.784     , 0.72      , 0.776     , 0.83870968, 0.75      ])